In [0]:
# Databricks notebook source
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [0]:
# 1. Enable MLflow Autologging
user_email = spark.sql("SELECT current_user()").collect()[0][0]
experiment_path = f"/Users/{user_email}/Bike_Customer_Churn_Experiment"
mlflow.set_experiment(experiment_path)

mlflow.autolog()

In [0]:
# 2. Load Features from Unity Catalog
df = spark.table("gold.ml_customer_features").toPandas()

In [0]:
# 3. Define Target Variable (Churn: Recency > 180 days = 1, else = 0)
df['is_churn'] = (df['recency'] > 180).astype(int)

In [0]:
# Select Predictor Features & Target
feature_cols = ['frequency', 'monetary', 'avg_order_value']
X = df[feature_cols]
y = df['is_churn']

In [0]:
# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [0]:
# 4. MLflow Run - Training & Registration
with mlflow.start_run(run_name="Random_Forest_Churn_Model") as run:
    params = {"n_estimators": 100, "max_depth": 6, "random_state": 42}
    
    # Train Model
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    
    # Evaluate Model
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Log Custom Metrics
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    
    # Log Model Artifact
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        input_example=X_train.head(3)
    )
    
    # Register Model to Unity Catalog Registry
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"
    model_name = "gold.customer_churn_model"
    
    registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)

print(f"Model Training Complete. Accuracy: {acc:.4f} | Registered as `{model_name}`.")